# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [18]:
from datetime import datetime
from agent import Agent

In [19]:
## TODO: Create the agent's instructions

ECOHOME_SYSTEM_PROMPT = """
You are EcoHome, an AI-powered energy optimization assistant for residential homes.

## Persona
You are a knowledgeable, practical, and trustworthy energy advisor. You help homeowners
understand their energy consumption, reduce costs, and make smarter decisions about
appliances, solar, and utility usage. You speak in a friendly but professional tone, and
always prioritize actionable advice over generic tips.

## Core Capabilities
- Analyze energy usage patterns from historical data
- Query real-time and forecasted electricity prices (time-of-use rates)
- Retrieve weather forecasts and solar irradiance data
- Search energy-saving best practices from a knowledge base
- Calculate potential savings from efficiency improvements
- Provide personalized recommendations ranked by impact and ease

## Available Tools
1. **query_energy_usage** — Get energy consumption data by date range and device type
2. **query_solar_generation** — Get solar production data by date range
3. **get_recent_energy_summary** — Quick 24h (or custom) usage + generation snapshot
4. **get_weather_forecast** — Weather forecast including solar irradiance estimates
5. **get_electricity_prices** — Time-of-use pricing for any date (peak/off-peak rates)
6. **search_energy_tips** — Semantic search through energy-saving best practices
7. **calculate_energy_savings** — Compute kWh/USD savings for optimization scenarios

## Constraints
- Always cite data sources when making claims (e.g., "Based on your usage last
month...")
- Never fabricate numbers — use actual tool outputs or clearly state estimates
- If critical data is missing (e.g., device wattage, tariff info), ask the user before
guessing
- Do not recommend unsafe actions (e.g., modifying electrical systems without a
professional)
- Respect user privacy — never store or share personal data beyond the current session
- If asked about topics outside energy (medical, legal, financial), politely redirect

## Response Best Practices
- Start with the most impactful recommendation, then detail the reasoning
- Include estimated savings (kWh and USD) when data supports it
- Explain tradeoffs (e.g., "This reducespeak usage but may require changing your
schedule")
- Use structured formatting: bullet points for tips, tables for comparisons, bold for
key numbers
- End with a brief summary of what the user should do next
- Always ask if the user wants to explore another area (solar, specific appliances, bill
breakdown)

## User Interaction Style
- Acknowledge the user's goals upfront before diving into analysis
- Use their home data whenever possible — personalization builds trust
- Propose concrete next steps, not just "let me know if you need help"
- If the user asks a vague question, clarify before answering (e.g., "Are you asking
about summer cooling costs or year-round usage?")

## Tone and Voice
- Helpful, encouraging, not judgmental
- Avoid jargon — explain terms like "demand charge" or "peak hours" when first used
- Be concise — respect the user's time

"""

In [20]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [21]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [22]:
print(response["messages"][-1].content)

Here's the optimal charging strategy for your electric car tomorrow (October 7, 2023) based on electricity prices and solar power generation:

### Electricity Prices (Time-of-Use Rates)
- **Off-Peak (Lowest Rate: $0.08/kWh)**: 
  - 12 AM - 5 AM
  - 10 PM - 11 PM
- **Part-Peak ($0.12/kWh)**: 
  - 6 AM - 9 AM
  - 2 PM - 4 PM
- **Peak (Highest Rate: $0.18/kWh)**: 
  - 10 AM - 1 PM
  - 4 PM - 9 PM

### Solar Power Generation Forecast
- **Solar Irradiance (W/m²)**:
  - **6 AM**: 288.2
  - **7 AM**: 594.4 (Peak solar generation starts)
  - **8 AM**: 447.8
  - **9 AM**: 568.2
  - **10 AM**: 277.1
  - **11 AM**: 383.4
  - **12 PM**: 206.6
  - **1 PM**: 520.7
  - **2 PM**: 365.3
  - **3 PM**: 168.6
  - **4 PM**: 473.9
  - **5 PM**: 238.1

### Recommended Charging Times
1. **Best Time to Charge**: 
   - **6 AM - 9 AM** (Part-Peak) 
     - This period has a good balance of solar generation starting to ramp up and lower electricity rates compared to peak hours.
2. **Alternative Time**: 
   - **12 

In [23]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_electricity_prices
- get_weather_forecast


## 2. Define Test Cases

In [ ]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [24]:
test1 = {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration",
        }

In [25]:
test2 = {
        "id": "thermostat_1",
        "question": "What's the best thermostat setting for summer to balance comfort and energy savings?",
        "expected_tools": ["search_energy_tips", "get_weather_forecast"],
        "expected_response": "Should include specific temperature recommendation, estimated savings, and reasoning",
    }

In [26]:
test3 = {
        "id": "appliance_scheduling_1",
        "question": "When is the best time to run my dishwasher and washing machine to save money?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "Should recommend off-peak hours and explain time-of-use pricing",
    }

In [27]:
test4 = {
        "id": "solar_optimization_1",
        "question": "How can I maximize my solar power usage during the day?",
        "expected_tools": ["get_weather_forecast", "query_solar_generation", "search_energy_tips"],
        "expected_response": "Should include timing recommendations for high-consumption appliances and solar production patterns",
    }

In [28]:
test5 = {
        "id": "cost_analysis_1",
        "question": "I used 500 kWh last month and my bill was $65. Is that reasonable for a 2-bedroom apartment?",
        "expected_tools": ["query_energy_usage", "get_electricity_prices"],
        "expected_response": "Should compare against average usage, explain rate tiers, and provide context",
    }

In [29]:
test6 = {
        "id": "hvac_1",
        "question": "My AC runs constantly during hot afternoons. What can I do to reduce energy use without sacrificing comfort?",
        "expected_tools": ["get_recent_energy_summary", "get_weather_forecast", "search_energy_tips"],
        "expected_response": "Should identify HVAC as high consumer and provide specific optimization tips",
    }

In [30]:
test7 = {
        "id": "peak_vs_offpeak_1",
        "question": "What's the difference between peak and off-peak electricity rates, and why does it matter for my bill?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "Should clearly explain time-of-use tiers, hours, and provide cost impact examples",
    }

In [31]:
test8 = {
        "id": "ev_charging_2",
        "question": "I have a Tesla Model 3 with 60 kWh battery. How much would it cost to fully charge at home?",
        "expected_tools": ["calculate_energy_savings", "get_electricity_prices"],
        "expected_response": "Should calculate cost based on current rates and explain variables (battery size, efficiency)",
    }

In [32]:
test9 = {
        "id": "pool_pump_1",
        "question": "I have a swimming pool with a 1.5 HP pump. What's the most energy-efficient schedule for running it?",
        "expected_tools": ["search_energy_tips", "get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "Should recommend off-peak scheduling, estimate daily/weekly cost, and discuss runtime optimization",
    }

In [33]:
test10 = {
        "id": "energy_audit_1",
        "question": "What are the biggest energy wasters in a typical home and what are the quick fixes I can do myself?",
        "expected_tools": ["search_energy_tips", "get_recent_energy_summary"],
        "expected_response": "Should list common culprits (phantom load, HVAC, lighting), provide actionable tips with estimated savings",
    }

In [34]:
test_cases = [ test1, test2, test3, test4, test5, test6, test7, test8, test9, test10 ]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [35]:
CONTEXT = "Location: San Francisco, CA"

In [36]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: thermostat_1
Question: What's the best thermostat setting for summer to balance comfort and energy savings?
--------------------------------------------------

Test 3: appliance_scheduling_1
Question: When is the best time to run my dishwasher and washing machine to save money?
--------------------------------------------------

Test 4: solar_optimization_1
Question: How can I maximize my solar power usage during the day?
--------------------------------------------------

Test 5: cost_analysis_1
Question: I used 500 kWh last month and my bill was $65. Is that reasonable for a 2-bedroom apartment?
--------------------------------------------------

Test 6: hvac_1
Question: My AC runs constantly during hot afternoons. What can I do to reduce energy use without sacrificing comfort?

In [37]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='dad9b753-e682-4279-96b4-b8fac76270ad'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='19209591-1f5e-4eec-849e-70c49bc4a293'),
    AIMessage(content="To provide you with the best charging times for your electric car tomorrow, I'll need to check the forecasted electricity prices and the solar generation potential for San Francisco, CA. This will help us identify the optimal charging window that aligns with lower electricity rates and higher solar production.\n\nLet me gather that information for you!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_toke

## 4. Evaluate Responses

In [ ]:
# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage

In [ ]:
# TODO: Create a response evaluator
def evaluate_response(question, final_response, expected_response):
    """Evaluate a single response against expected response"""
    pass

In [ ]:
# TODO: Create a tool udage evaluator
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate if the right tools were used"""
    pass

In [ ]:
# TODO: Generate a comprehensive evaluation report
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement
def generate_evaluation_report():
    pass